In [1]:
!pip -q install -U bitsandbytes accelerate "transformers>=4.56.2" "trl==0.27.1" peft datasets


In [2]:
!python --version
!nvidia-smi

import torch
print("torch:", torch.__version__)
print("cuda available:", torch.cuda.is_available())
print("torch cuda:", torch.version.cuda)
if torch.cuda.is_available():
    print("gpu:", torch.cuda.get_device_name(0))

try:
    import bitsandbytes as bnb
    print("bitsandbytes:", bnb.__version__)
except Exception as e:
    print("bitsandbytes import FAIL:", repr(e))


Python 3.12.12
Sat Jan 24 20:42:44 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.54.15              Driver Version: 550.54.15      CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   60C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+--------------------------------

In [21]:
import json
import numpy as np
import pathlib
import os
import argparse
import re
from typing import Iterable, List, Dict, Any
from peft import PeftModel
from datasets import Dataset, load_dataset
import torch
from datasets import load_dataset
from peft import get_peft_model, LoraConfig, prepare_model_for_kbit_training
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, TrainingArguments, Trainer, DataCollatorForLanguageModeling, pipeline
from trl import SFTConfig, SFTTrainer


In [4]:
import transformers, accelerate, bitsandbytes as bnb

print("py", __import__("sys").version.split()[0])
print("torch", torch.__version__, "cuda", torch.version.cuda, "avail", torch.cuda.is_available())
print("transformers", transformers.__version__)
print("accelerate", accelerate.__version__)
print("bnb", bnb.__version__)


py 3.12.12
torch 2.9.0+cu126 cuda 12.6 avail True
transformers 4.57.6
accelerate 1.12.0
bnb 0.49.1


In [18]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [5]:
base = "json_files"
files = [
    f"{base}/ehac262_clean.json",
    f"{base}/ehae176.json",
    f"{base}/ehae178.json",
    f"{base}/ehaf190.json",
    f"{base}/ehaf194.json",
]

In [6]:
ds = load_dataset("json", data_files=files, split="train")
print(ds)

Dataset({
    features: ['page', 'text'],
    num_rows: 310
})


In [7]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
model_name = "mistralai/Mistral-7B-v0.3"

tokenizer = AutoTokenizer.from_pretrained(model_name)
if tokenizer.pad_token_id is None:
    tokenizer.pad_token_id = tokenizer.eos_token_id

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [8]:
def tokenize(batch):
    return tokenizer(
        batch["text"],
        add_special_tokens=True,
        return_attention_mask=True,
        #Ako nemaš truncation=True tokenizer će vratiti cijeli input (koliko god bio dug).
        truncation=True,
        max_length=4096,
    )

tokenized_ds = ds.map(tokenize, batched=True, remove_columns=ds.column_names)


Map:   0%|          | 0/310 [00:00<?, ? examples/s]

In [15]:
print(ds[0])
print(tokenized_ds[0])
print(tokenized_ds.column_names)
print(tokenized_ds[0].keys())

lens = np.array([len(x) for x in tokenized_ds["input_ids"]])

print("chunks:", len(lens))
print("min:", lens.min())
print("p50:", int(np.percentile(lens, 50)))
print("p80:", int(np.percentile(lens, 80)))
print("p90:", int(np.percentile(lens, 90)))
print("p95:", int(np.percentile(lens, 95)))
print("p99:", int(np.percentile(lens, 99)))
print("max:", lens.max())

#treba se urediti dataset da max bude ispod 2048 po mogucnosti, iako je skoro 99 posto ds ujutar tog rangea
#4096 je preveliko cak i za 3418
#mozemo samo izabrati 2048 i pustit da se neki dijelovi odsijeku


{'page': 4002, 'text': 'list\nAbbreviations and acronyms\nAAD Anti-arrhythmic drug\nACE-I Angiotensin-converting enzyme inhibitor\nACS Acute coronary syndrome\nAED Automated external defibrillator\nAF Atrial fibrillation\nAH Atrial–His interval\nALS Advanced life support\nARB Angiotensin receptor blocker\nARNI Angiotensin receptor neprilysin inhibitor\nARVC Arrhythmogenic right ventricular cardiomyopathy\nATP Anti-tachycardia pacing\nAV Atrioventricular\nAVRT AV re-entry tachycardia\nBBR-VT Bundle branch re-entrant ventricular tachycardia\nBrS Brugada syndrome\nCA Cardiac arrest\ncAMP Cyclic adenosine monophosphate\nCAD Coronary artery disease\nCAG Coronary angiogram\nCCB Calcium channel blocker\nCHD Congenital heart disease\nCIED Cardiac implantable electronic devices\nCMR Cardiac magnetic resonance\nCPR Cardiopulmonary resuscitation\nCPVT Catecholaminergic polymorphic ventricular tachycardia\nCRT Cardiac resynchronization therapy\nCT/CTA Computed tomography/Computed tomography angiog

In [10]:
compute_dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=compute_dtype,
)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto",
    dtype=compute_dtype,
)

model.config.use_cache = False  # obavezno za trening


Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

In [11]:

lora_config = LoraConfig(
    r=8, #Veći r = više trenirajućih parametara = veća sposobnost učenja, ali više VRAM-a i sporije.
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"] ,#moduli koje ćemo prilagoditi LoR-om (quey, key, value, output projekcije u attention slojevima)
    lora_alpha=32, #veći alpha = veća težina LoRa prilagodbe u odnosu na osnovne težine.LoRA “jače utječe” (brže uči, ali može biti nestabilnije)
    lora_dropout=0.05, #regularizacija
    bias="none",
    task_type="CAUSAL_LM",

)


In [12]:
model = prepare_model_for_kbit_training(model)
model.gradient_checkpointing_enable()

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

trainable params: 6,815,744 || all params: 7,254,839,296 || trainable%: 0.0939


In [14]:
data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

model.config.use_cache = False

args = TrainingArguments(
    output_dir="./mistral_lora_out",
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    num_train_epochs=1,
    learning_rate=2e-4,
    fp16=True,
    logging_strategy="steps",
    logging_steps=1,
    save_steps=200,
    report_to="none",
    remove_unused_columns=False,
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=tokenized_ds,
    data_collator=data_collator,
)

trainer.train()


Step,Training Loss
1,1.406100
2,1.427200
3,1.422200
4,1.562600
5,1.326900
6,1.299000
7,1.481300
8,1.323400
9,1.636600
10,1.233600


TrainOutput(global_step=39, training_loss=1.3760904226547632, metrics={'train_runtime': 1687.5108, 'train_samples_per_second': 0.184, 'train_steps_per_second': 0.023, 'total_flos': 1.5672388382466048e+16, 'train_loss': 1.3760904226547632, 'epoch': 1.0})

In [16]:
out_dir = "./mistral_lora_out"
trainer.model.save_pretrained(out_dir)
tokenizer.save_pretrained(out_dir)
print("Saved adapter + tokenizer to:", out_dir)

Saved adapter + tokenizer to: ./mistral_lora_out


In [17]:
!zip -r mistral_lora_out.zip /content/mistral_lora_out


  adding: content/mistral_lora_out/ (stored 0%)
  adding: content/mistral_lora_out/tokenizer_config.json (deflated 96%)
  adding: content/mistral_lora_out/adapter_config.json (deflated 57%)
  adding: content/mistral_lora_out/README.md (deflated 66%)
  adding: content/mistral_lora_out/checkpoint-39/ (stored 0%)
  adding: content/mistral_lora_out/checkpoint-39/tokenizer_config.json (deflated 96%)
  adding: content/mistral_lora_out/checkpoint-39/adapter_config.json (deflated 57%)
  adding: content/mistral_lora_out/checkpoint-39/README.md (deflated 66%)
  adding: content/mistral_lora_out/checkpoint-39/optimizer.pt (deflated 8%)
  adding: content/mistral_lora_out/checkpoint-39/scheduler.pt (deflated 62%)
  adding: content/mistral_lora_out/checkpoint-39/rng_state.pth (deflated 26%)
  adding: content/mistral_lora_out/checkpoint-39/training_args.bin (deflated 53%)
  adding: content/mistral_lora_out/checkpoint-39/trainer_state.json (deflated 79%)
  adding: content/mistral_lora_out/checkpoint-39

In [19]:
!mkdir -p "/content/drive/MyDrive/mistral_lora_out"
!cp -r "/content/mistral_lora_out/"* "/content/drive/MyDrive/mistral_lora_out/"
!ls -lah "/content/drive/MyDrive/mistral_lora_out"


total 31M
-rw------- 1 root root 1006 Jan 24 21:28 adapter_config.json
-rw------- 1 root root  27M Jan 24 21:28 adapter_model.safetensors
drwx------ 2 root root 4.0K Jan 24 21:28 checkpoint-39
-rw------- 1 root root 5.1K Jan 24 21:28 README.md
-rw------- 1 root root  437 Jan 24 21:28 special_tokens_map.json
-rw------- 1 root root 134K Jan 24 21:28 tokenizer_config.json
-rw------- 1 root root 3.6M Jan 24 21:28 tokenizer.json
-rw------- 1 root root 574K Jan 24 21:28 tokenizer.model


Testiranje

In [22]:
model_name = "mistralai/Mistral-7B-v0.3"
adapter_dir = "./mistral_lora_out"

compute_dtype = torch.float16  # T4
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=compute_dtype,
)

tokenizer = AutoTokenizer.from_pretrained(adapter_dir, use_fast=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

base = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto",
    dtype=compute_dtype,
)

lora = PeftModel.from_pretrained(base, adapter_dir)
lora.eval()

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

MistralForCausalLM(
  (model): MistralModel(
    (embed_tokens): Embedding(32768, 4096)
    (layers): ModuleList(
      (0-31): 32 x MistralDecoderLayer(
        (self_attn): MistralAttention(
          (q_proj): lora.Linear4bit(
            (base_layer): Linear4bit(in_features=4096, out_features=4096, bias=False)
            (lora_dropout): ModuleDict(
              (default): Dropout(p=0.05, inplace=False)
            )
            (lora_A): ModuleDict(
              (default): Linear(in_features=4096, out_features=8, bias=False)
            )
            (lora_B): ModuleDict(
              (default): Linear(in_features=8, out_features=4096, bias=False)
            )
            (lora_embedding_A): ParameterDict()
            (lora_embedding_B): ParameterDict()
            (lora_magnitude_vector): ModuleDict()
          )
          (k_proj): lora.Linear4bit(
            (base_layer): Linear4bit(in_features=4096, out_features=1024, bias=False)
            (lora_dropout): ModuleDict(
 

In [ ]:
def ask_once(model, question: str, max_new_tokens: int = 120) -> str:
    prompt = (
        "Answer in 1-2 sentences. Do not add new questions.\n"
        f"Q: {question}\nA:"
    )
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    prompt_len = inputs["input_ids"].shape[1]

    with torch.no_grad():
        out = model.generate(
            **inputs,
            do_sample=False,
            max_new_tokens=max_new_tokens,
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )

    # decode ONLY the newly generated tokens (without the prompt)
    gen_tokens = out[0][prompt_len:]
    gen_text = tokenizer.decode(gen_tokens, skip_special_tokens=True)

    # if it starts writing a new question, cut it off
    cut = gen_text.find("\nQ:")
    if cut != -1:
        gen_text = gen_text[:cut]

    return gen_text.strip()


# --- tests ---
tests = [
    "What does LCSD stand for and when is it indicated in LQTS?",
    "List key recommendations for patients with Brugada syndrome (general measures).",
    "Define CPVT and state first-line therapy.",
    "What is an ICD and when is implantation not recommended?",
    "Summarize the purpose of genetic testing in inherited arrhythmia syndromes.",
]

for q in tests:
    print("\n" + "=" * 90)
    print("Q:", q)

    print("\n--- BASE (adapter OFF) ---")
    with lora.disable_adapter():
        print(ask_once(lora, q))

    print("\n--- LORA (adapter ON) ---")
    print(ask_once(lora, q))


Q: What does LCSD stand for and when is it indicated in LQTS?

--- BASE (adapter OFF) ---
Long QT Syndrome

--- LORA (adapter ON) ---
Long QT syndrome diagnosis and management.

Q: List key recommendations for patients with Brugada syndrome (general measures).

--- BASE (adapter OFF) ---
- Avoid drugs that prolong the QT interval.
- Avoid drugs that cause hypokalemia.
- Avoid drugs that cause hypomagnesemia.
- Avoid drugs that cause hypocalcemia.
- Avoid drugs that cause hypothyroidism.
- Avoid drugs that cause hypothermia.
- Avoid drugs that cause hyperthermia.
- Avoid drugs that cause hypoxia.
- Avoid drugs that cause hyperglycemia.
- Avoid drugs that cause hyperinsulinemia.
- Avoid drugs that

--- LORA (adapter ON) ---


Q: Define CPVT and state first-line therapy.

--- BASE (adapter OFF) ---
Cardiac potassium channelopathies are a group of inherited disorders that affect the cardiac action potential. CPVT is a rare genetic disorder that causes sudden cardiac death in young people.